# 1.Ответы на вопросы

I. Аналитическое решение аналитической регрессии:  
Минимизируем функцию потерь:  
$$
L(w) = \|Xw - y|^2
$$  
Производная по w:  
$$
\frac{\partial L}{\partial w} = 2 X^{T} (Xw - y)
$$
Приравниваем к нулю:  
$$
X^{T} X w = X^{T} y
$$  
Итог:  
$$
w = (X^{T} X)^{-1} X^{T} y
$$

II. При регуляризации добавляется штраф к функции потерь:
При L2:  
$\large L = (y - \hat{y})^2 = (y - Xw)(y - Xw)^T + \lambda \|x\|_2^2$  

$\large \frac{dL}{dw} = -2X^T(y - Xw) + 2\lambda w$  

$\large L = (y - \hat{y})^2 = (y - Xw)(y - Xw)^T + \lambda \|x\|_1$  

При L1:  
  
$\large \frac{dL}{dw} = -2X^T(y - Xw) + \lambda \operatorname{sign}(w)$  

III. L1 сжимает крупные признаки, а маленькие становяться нулем, так она отбрасывает ненужные признаки и оставляет только важные

IV. Для приближения нелинейных зависимостей нужно преобразовать признаки, например, добавить полиномиальные признаки или преобразование в логарифмические функции, корень, экспонента.

# 2. Введение

Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sklearn
import lightgbm
import scipy
import statsmodels 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression , Lasso , Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import re
from collections import Counter

In [2]:
train = pd.read_json('../datasets/train.json')
test = pd.read_json('../datasets/train.json')
test

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
4,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[Dining Room, Pre-War, Laundry in Building, Di...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,medium
6,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, Laundry in Building, Dishw...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,low
9,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, Laundry in Building, Laund...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,medium
10,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,medium
15,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, Fitness Center, Laundry in...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124000,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, Hardwood Floors]",40.8433,6824800,-73.9396,a61e21da3ba18c7a3d54cfdcc247e1f8,[https://photos.renthop.com/2/6824800_0682be16...,2800,620 W 171 Street,low
124002,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[Common Outdoor Space, Cats Allowed, Dogs Allo...",40.8198,6813268,-73.9578,8f90e5e10e8a2d7cf997f016d89230eb,[https://photos.renthop.com/2/6813268_1e6fcc32...,2395,3333 Broadway,medium
124004,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[Dining Room, Elevator, Pre-War, Laundry in Bu...",40.5765,6927093,-73.9554,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/6927093_93a52104...,1850,210 Brighton 15th St,medium
124008,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, Laundry in Unit, Dishwasher, No Fee,...",40.7448,6892816,-74.0017,c3cd45f4381ac371507090e9ffabea80,[https://photos.renthop.com/2/6892816_1a8d087a...,4195,350 West 21st Street,medium


In [3]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()
train['interest_level'] = label.fit_transform(train['interest_level'])


###encodes = {'low': 1, 'medium': 2,'high': 3}
###train['interest_level'] = train['interest_level'].map(encodes)


In [4]:
train['features'] = train['features'].apply(
    lambda lst: [re.sub(r'[\s\'"]', '', f) for f in lst if isinstance(f, str)]
)
train = train[(train["price"] >= train["price"].quantile(0.01)) & 
                                (train["price"] <= train["price"].quantile(0.99))]

train = train.reset_index(drop=True)
train

,bathrooms,bedrooms,building_id,created,description,display_address,features,latitude,listing_id,longitude,manager_id,photos,price,street_address,interest_level
0,1.0,1,8579a0b0d54db803821a35a4a615e97a,2016-06-16 05:55:27,Spacious 1 Bedroom 1 Bathroom in Williamsburg!...,145 Borinquen Place,"[DiningRoom, Pre-War, LaundryinBuilding, Dishw...",40.7108,7170325,-73.9539,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/7170325_3bb5ac84...,2400,145 Borinquen Place,2
1,1.0,2,b8e75fc949a6cd8225b455648a951712,2016-06-01 05:44:33,BRAND NEW GUT RENOVATED TRUE 2 BEDROOMFind you...,East 44th,"[Doorman, Elevator, LaundryinBuilding, Dishwas...",40.7513,7092344,-73.9722,955db33477af4f40004820b4aed804a0,[https://photos.renthop.com/2/7092344_7663c19a...,3800,230 East 44th,1
2,1.0,2,cd759a988b8f23924b5a2058d5ab2b49,2016-06-14 15:19:59,**FLEX 2 BEDROOM WITH FULL PRESSURIZED WALL**L...,East 56th Street,"[Doorman, Elevator, LaundryinBuilding, Laundry...",40.7575,7158677,-73.9625,c8b10a317b766204f08e613cef4ce7a0,[https://photos.renthop.com/2/7158677_c897a134...,3495,405 East 56th Street,2
3,1.5,3,53a5b119ba8f7b61d4e010512e0dfc85,2016-06-24 07:54:24,A Brand New 3 Bedroom 1.5 bath ApartmentEnjoy ...,Metropolitan Avenue,[],40.7145,7211212,-73.9425,5ba989232d0489da1b5f2c45f6688adc,[https://photos.renthop.com/2/7211212_1ed4542e...,3000,792 Metropolitan Avenue,2
4,1.0,0,bfb9405149bfff42a92980b594c28234,2016-06-28 03:50:23,Over-sized Studio w abundant closets. Availabl...,East 34th Street,"[Doorman, Elevator, FitnessCenter, LaundryinBu...",40.7439,7225292,-73.9743,2c3b41f588fbb5234d8a1e885a436cfa,[https://photos.renthop.com/2/7225292_901f1984...,2795,340 East 34th Street,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48374,1.0,3,92bbbf38baadfde0576fc496bd41749c,2016-04-05 03:58:33,There is 700 square feet of recently renovated...,W 171 Street,"[Elevator, Dishwasher, HardwoodFloors]",40.8433,6824800,-73.9396,a61e21da3ba18c7a3d54cfdcc247e1f8,[https://photos.renthop.com/2/6824800_0682be16...,2800,620 W 171 Street,1
48375,1.0,2,5565db9b7cba3603834c4aa6f2950960,2016-04-02 02:25:31,"2 bedroom apartment with updated kitchen, rece...",Broadway,"[CommonOutdoorSpace, CatsAllowed, DogsAllowed,...",40.8198,6813268,-73.9578,8f90e5e10e8a2d7cf997f016d89230eb,[https://photos.renthop.com/2/6813268_1e6fcc32...,2395,3333 Broadway,2
48376,1.0,1,67997a128056ee1ed7d046bbb856e3c7,2016-04-26 05:42:03,No Brokers Fee * Never Lived 1 Bedroom 1 Bathr...,210 Brighton 15th St,"[DiningRoom, Elevator, Pre-War, LaundryinBuild...",40.5765,6927093,-73.9554,a10db4590843d78c784171a107bdacb4,[https://photos.renthop.com/2/6927093_93a52104...,1850,210 Brighton 15th St,2
48377,1.0,2,3c0574a740154806c18bdf1fddd3d966,2016-04-19 02:47:33,Wonderful Bright Chelsea 2 Bedroom apartment o...,West 21st Street,"[Pre-War, LaundryinUnit, Dishwasher, NoFee, Ou...",40.7448,6892816,-74.0017,c3cd45f4381ac371507090e9ffabea80,[https://photos.renthop.com/2/6892816_1a8d087a...,4195,350 West 21st Street,2


In [5]:
all_features = [f for sublist in train['features'] for f in sublist]
features_cnt = Counter(all_features)
all_features

['DiningRoom',
 'Pre-War',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'DogsAllowed',
 'CatsAllowed',
 'Doorman',
 'Elevator',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'Doorman',
 'Elevator',
 'LaundryinBuilding',
 'LaundryinUnit',
 'Dishwasher',
 'HardwoodFloors',
 'Doorman',
 'Elevator',
 'FitnessCenter',
 'LaundryinBuilding',
 'Doorman',
 'Elevator',
 'Loft',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'Fireplace',
 'LaundryinUnit',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'Elevator',
 'LaundryinBuilding',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'HardwoodFloors',
 'CatsAllowed',
 'DogsAllowed',
 'Doorman',
 'Elevator',
 'LaundryinBuilding',
 'DogsAllowed',
 'CatsAllowed',
 'RoofDeck',
 'Doorman',
 'Elevator',
 'FitnessCenter',
 'Pre-War',
 'LaundryinBuilding',
 'HighSpeedInternet',
 'Dishwasher',
 'HardwoodFloors',
 'NoFee',
 'DogsAllowed',
 'CatsAllowed',
 'SwimmingPool',
 'RoofDeck',
 'Doorman',
 'Elevator',
 'FitnessCenter',
 'Laun

In [6]:
unique_features = set(all_features)
len(unique_features)

1528

In [7]:
all = Counter(all_features)
[x[0] for x in all.most_common(21) ]
all.most_common(21)

[('Elevator', 25398),
 ('HardwoodFloors', 23159),
 ('CatsAllowed', 23148),
 ('DogsAllowed', 21662),
 ('Doorman', 20497),
 ('Dishwasher', 20095),
 ('NoFee', 17806),
 ('LaundryinBuilding', 16093),
 ('FitnessCenter', 13000),
 ('Pre-War', 8978),
 ('LaundryinUnit', 8448),
 ('RoofDeck', 6423),
 ('OutdoorSpace', 5137),
 ('DiningRoom', 4901),
 ('HighSpeedInternet', 4225),
 ('Balcony', 2898),
 ('SwimmingPool', 2648),
 ('LaundryInBuilding', 2565),
 ('NewConstruction', 2507),
 ('Terrace', 2179),
 ('Exclusive', 2080)]

In [8]:
features_cnt = Counter(all_features)
top_20_features = [feature[0] for feature in features_cnt.most_common(20)]
features_cnt.most_common(21)

[('Elevator', 25398),
 ('HardwoodFloors', 23159),
 ('CatsAllowed', 23148),
 ('DogsAllowed', 21662),
 ('Doorman', 20497),
 ('Dishwasher', 20095),
 ('NoFee', 17806),
 ('LaundryinBuilding', 16093),
 ('FitnessCenter', 13000),
 ('Pre-War', 8978),
 ('LaundryinUnit', 8448),
 ('RoofDeck', 6423),
 ('OutdoorSpace', 5137),
 ('DiningRoom', 4901),
 ('HighSpeedInternet', 4225),
 ('Balcony', 2898),
 ('SwimmingPool', 2648),
 ('LaundryInBuilding', 2565),
 ('NewConstruction', 2507),
 ('Terrace', 2179),
 ('Exclusive', 2080)]

In [9]:
for feature in top_20_features:
    train[feature] = train['features'].apply(lambda x: 1 if feature in x else 0)
    test[feature] = test['features'].apply(lambda x: 1 if feature in x else 0)

In [10]:
final_feature_list = ['bathrooms', 'bedrooms'] + top_20_features
final_feature_list

['bathrooms',
 'bedrooms',
 'Elevator',
 'HardwoodFloors',
 'CatsAllowed',
 'DogsAllowed',
 'Doorman',
 'Dishwasher',
 'NoFee',
 'LaundryinBuilding',
 'FitnessCenter',
 'Pre-War',
 'LaundryinUnit',
 'RoofDeck',
 'OutdoorSpace',
 'DiningRoom',
 'HighSpeedInternet',
 'Balcony',
 'SwimmingPool',
 'LaundryInBuilding',
 'NewConstruction',
 'Terrace']

In [11]:
X_train = train[final_feature_list]
y_train = train['price']
X_train
X_test = test[final_feature_list]
y_test = test['price']

In [13]:
class regression():
    def __init__(self , method = 'gd' , lr = 0.001, iter = 109 , seed = 21 , batch_size = 32 ):
        pass 

    def linearreg(self , x , y):
        k = x.shape[0]
        wb = np.c_[np.ones(k,1),x]
        self.w = ( wb @ wb.T ) @ wb.T @ y

    def grad(self,x,y):
        k = x.shape[0]
        wb = np.c_[np.ones(k,1),x]
        self.w = np.zeros(xb.shape[1])
        for i in range(self.iter):
            y_pred = xb @ self.w 
            grad = (2/n) * ( xb.T @ (_pred - y))
            self.w -= sef.lr * grad 
         

In [ ]:
class MyLinearRegression():
    def __init__(self,method='sgd', learning_rate=0.001, epochs=100, seed=21, batch_size=32):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.seed = seed
        self.w = None
        self.method = method
        self.batch_size = batch_size

    def _fit_analytic(self, X, y):
        k = X.shape[0]
        Xb = np.c_[np.ones((k, 1)), X]
        self.w = np.linalg.pinv(Xb.T @ Xb) @ Xb.T @ y
    
    def _fit_gd(self, X, y):
        np.random.seed(self.seed)
        Xb = np.c_[np.ones((X.shape[0], 1)), X]
        self.w = np.zeros(Xb.shape[1])
        n = Xb.shape[0]

        for i in range(self.epochs):
            y_pred = Xb @ self.w
            grad = (2 / n) * (Xb.T @ (y_pred - y))
            self.w -= self.learning_rate * grad

    def _fit_sgd(self, X, y):
        np.random.seed(self.seed)

        Xb = np.c_[np.ones((X.shape[0], 1)), X]
        self.w = np.zeros(Xb.shape[1], dtype=np.float64)
        n = Xb.shape[0]

        for epoch in range(self.epochs):
            indices = np.random.permutation(n)
            y_shuffled = y[indices]
            Xb_shuffled = Xb[indices]

            for i in range(0, n, self.batch_size):
                X_batch = Xb_shuffled[i: i+self.batch_size]
                y_batch = y_shuffled[i: i+self.batch_size]
                y_pred = X_batch @ self.w
                grad = (2 / len(X_batch)) * (X_batch.T @ (y_pred - y_batch))
                self.w -= self.learning_rate * grad
        
    def fit(self, X, y):
        if self.method == 'analytic':
            self._fit_analytic(X, y)
        elif self.method == 'gd':
            self._fit_gd(X, y)
        elif self.method == 'sgd':
            self._fit_sgd(X, y)

    def predict(self,X):
        X = np.c_[np.ones((X.shape[0], 1)), X]
        return X.dot(self.w)


In [ ]:
lr2 = MyLinearRegression(method='gd')
lr2.fit(X_train, y_train)
print(lr2.w)

[378.51209888 511.13932169 653.6917622  213.60099017 176.2636971
 180.05841015 169.53908069 186.04675871 165.80247914 137.22366268
 126.1837628  117.17267221  65.60316385  82.05098197  51.69608875
  44.03591924  47.04087484  32.07810992  26.2559678   24.82309966
  18.19940212  19.85466665  20.86892654]


In [ ]:
lr1 = MyLinearRegression(method='analytic')
lr1.fit(X_train, y_train)
print(lr1.w)

[ 596.43630905 1576.096428    463.07170317  228.44171762 -174.44931437
  -82.19950129  155.22893686  617.50240763  149.69180477 -163.55802212
 -220.38134739  242.70306501  -26.56830022  456.5541576  -104.07102444
  -91.63958952  121.20626186 -229.49115183  -38.87694636   34.64322972
 -364.0110301  -114.5752959   175.23649143]


In [ ]:
model = MyLinearRegression()
model.fit(X_train, y_train)
model.w

array([ 595.79383344, 1575.86696002,  463.79150185,  227.88968838,
       -176.33109981,  -79.94847525,  155.29197432,  616.61890034,
        147.98582383, -164.38329852, -222.45070249,  241.44264881,
        -26.74376982,  455.70027705, -104.82283461,  -93.06881412,
        120.33868688, -230.00529892,  -39.49872737,   34.25286755,
       -363.81492594, -115.09654633,  174.10619868])

In [ ]:
import numpy as np

def r2_metric(y_true, y_pred):
    y_true = np.array(y_true).reshape((-1, 1)) 
    y_pred = np.array(y_pred).reshape((-1, 1)) 
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    if ss_tot == 0:
        return 1.0 
    return 1 - (ss_res / ss_tot)

def rmse_metric(y_true, y_pred):
    y_true = np.array(y_true).reshape((-1, 1))
    y_pred = np.array(y_pred).reshape((-1, 1))
    n = len(y_true)
    mse = np.sum((y_true - y_pred)**2) / n
    return np.sqrt(mse)

def mae_metric(y_true, y_pred):
    y_true = np.array(y_true).reshape((-1, 1))
    y_pred = np.array(y_pred).reshape((-1, 1))
    n = len(y_true)
    mae = np.sum(np.abs(y_true - y_pred)) / n
    return mae

In [ ]:
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

In [ ]:
print(f"R2: {r2_metric(y_true=y_train, y_pred=y_train_pred)}")
print(f"MAE: {mae_metric(y_train, y_train_pred)}")
print(f"RMSE: {rmse_metric(y_train, y_train_pred)}")

R2: 0.580023606896459
MAE: 711.3314420705492
RMSE: 1035.3642715622052


In [ ]:
result_r2 = pd.DataFrame(columns=['model','train','test'])
result_mae = pd.DataFrame(columns=['model','train','test'])
result_rmse = pd.DataFrame(columns=['model','train','test'])

In [ ]:
result_r2.loc[1] = ['sgd', r2_metric(y_true=y_train, y_pred=y_train_pred), r2_metric(y_true=y_test, y_pred=y_test_pred)]
result_r2

,model,train,test
1,sgd,0.580024,0.580024


In [ ]:
result_mae.loc[1] = ['sgd', mae_metric(y_train, y_train_pred), mae_metric(y_test, y_test_pred)]
result_mae

,model,train,test
1,sgd,711.331442,711.331442


In [ ]:
result_rmse.loc[1] = ['sgd', rmse_metric(y_train, y_train_pred), rmse_metric(y_test, y_test_pred)]
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272


In [ ]:
from sklearn.linear_model import LinearRegression
model_lib = LinearRegression().fit(X_train, y_train)
y_lib_train_pred = model_lib.predict(X_train)
r2lib = r2_metric(y_true=y_train, y_pred=y_lib_train_pred)
maelib = mae_metric(y_true=y_train, y_pred=y_lib_train_pred)
rmselib = rmse_metric(y_true=y_train, y_pred=y_lib_train_pred)
print(r2lib)
print(maelib)
print(rmselib)
y_lib_test_pred = model_lib.predict(X_test)
r2libtest = r2_metric(y_true=y_test, y_pred=y_lib_test_pred)
maelibtest = mae_metric(y_true=y_test, y_pred=y_lib_test_pred)
rmselibtest = rmse_metric(y_true=y_test, y_pred=y_lib_test_pred)
print(r2libtest)
print(maelibtest)
print(rmselibtest)

0.5800339065557949
711.7911655149248
1035.3515756525746
0.5800339065557949
711.7911655149248
1035.3515756525746


In [ ]:
class MyLinearRegressionRegulized():
    def __init__(self, learning_rate=0.001, epochs=100, seed=21, batch_size=32, algorithm='Ridge', alpha=0.1, el_coef=0.5):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.seed = seed
        self.w = None
        self.batch_size = batch_size
        self.algorithm = algorithm.lower()
        self.alpha = alpha
        self.el_coef = el_coef
    
    def _fit_sgd(self, X, y):
        np.random.seed(self.seed)

        Xb = np.c_[np.ones((X.shape[0], 1)), X]
        self.w = np.zeros(Xb.shape[1], dtype=np.float64)
        n = Xb.shape[0]

        for epoch in range(self.epochs):
            indices = np.random.permutation(n)
            y_shuffled = y[indices]
            Xb_shuffled = Xb[indices]

            for i in range(0, n, self.batch_size):
                X_batch = Xb_shuffled[i: i+self.batch_size]
                y_batch = y_shuffled[i: i+self.batch_size]
                y_pred = X_batch @ self.w
                grad = (2 / len(X_batch)) * (X_batch.T @ (y_pred - y_batch))
                
                w_reg = self.w.copy()
                w_reg[0] = 0

                if self.algorithm == 'l2' or self.algorithm == 'ridge':
                    grad += 2 * self.alpha * w_reg
                elif self.algorithm == 'l1' or self.algorithm == 'lasso':
                    grad += self.alpha * np.sign(w_reg)
                elif self.algorithm == 'elasticnet':
                    grad += self.alpha * (self.el_coef * np.sign(w_reg) + (1 - self.el_coef) * 2 * w_reg)
                else:
                    print('Такой способ отсутсвует')
                    return
                self.w -= self.learning_rate * grad

    def fit(self, X, y):
        self._fit_sgd(X, y)

    def predict(self, X):
        X = np.c_[np.ones((X.shape[0], 1)), X]
        return X.dot(self.w)
                

In [ ]:
reg_model = MyLinearRegressionRegulized()
reg_model.fit(X_train, y_train)
y_ridge_pred = reg_model.predict(X_train)
y_ridge_pred_test = reg_model.predict(X_test)
reg_model.w

array([1137.84932278, 1023.42495253,  529.30115365,  202.37998176,
       -100.57059265,   28.47228013,   65.57581204,  431.14037591,
        118.48342242,  -93.73685205,  -82.85864076,  196.27038263,
        -29.01620502,  287.18474646,  -40.56672117,    2.24420868,
        104.92683429,  -89.91031409,   23.986781  ,   48.14558669,
       -101.28647191,  -29.99568319,   57.84181695])

In [ ]:
r2ridge = r2_metric(y_true=y_train, y_pred=y_ridge_pred)

In [ ]:
maeridge = mae_metric(y_true=y_train, y_pred=y_ridge_pred)
maeridge

np.float64(728.2168959411032)

In [ ]:
rmseridge= rmse_metric(y_true=y_train, y_pred=y_ridge_pred)
rmseridge

np.float64(1071.1509637660936)

In [ ]:
from sklearn.linear_model import Ridge
sklear_ridge = Ridge(fit_intercept=True)
sklear_ridge.fit(X_train, y_train)
y_sklear_ridge_pred_train = sklear_ridge.predict(X_train)
r2_sklearn_ridge = r2_metric(y_true=y_train, y_pred=y_sklear_ridge_pred_train)
r2_sklearn_ridge

np.float64(0.580033902304394)

In [ ]:
mae_sklearn_ridge = mae_metric(y_true=y_train, y_pred=y_sklear_ridge_pred_train)
mae_sklearn_ridge

np.float64(711.787958224175)

In [ ]:
rmse_sklearn_ridge = rmse_metric(y_true=y_train, y_pred=y_sklear_ridge_pred_train)
rmse_sklearn_ridge

np.float64(1035.3515808931104)

In [ ]:
y_sklear_ridge_pred_test = sklear_ridge.predict(X_test)
print(r2_metric(y_test, y_ridge_pred_test))
print(r2_metric(y_test, y_sklear_ridge_pred_test))

0.5504894421297051
0.580033902304394


In [ ]:
print(mae_metric(y_test, y_ridge_pred_test))
print(mae_metric(y_test, y_sklear_ridge_pred_test))


728.2168959411032
711.787958224175


In [ ]:
print(rmse_metric(y_test, y_ridge_pred_test))
print(rmse_metric(y_test, y_sklear_ridge_pred_test))
result_r2.loc[2] = ['Ridge', r2ridge, r2_metric(y_test, y_ridge_pred_test)]
result_mae.loc[2] = ['Ridge', maeridge, mae_metric(y_test, y_ridge_pred_test)]
result_rmse.loc[2] = ['Ridge', rmseridge, rmse_metric(y_test, y_ridge_pred_test)]
result_rmse

1071.1509637660936
1035.3515808931104


,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964


In [ ]:
reg_model_l1 = MyLinearRegressionRegulized(algorithm='l1')
reg_model_l1.fit(X_train, y_train)
l1_train_pred = reg_model_l1.predict(X_train)
l1_test_pred = reg_model_l1.predict(X_test)
from sklearn.linear_model import Lasso
sklearn_lasso = Lasso()
sklearn_lasso.fit(X_train, y_train)
sklearn_lasso_pred_train = sklearn_lasso.predict(X_train)
sklearn_lasso_pred_test = sklearn_lasso.predict(X_test)

In [ ]:
r2lasoo = r2_metric(y_true=y_train, y_pred=l1_train_pred)
libr2lasso = r2_metric(y_true=y_train, y_pred=sklearn_lasso_pred_train)
print(r2lasoo)
print(libr2lasso)

0.5800231726410873
0.5798741149197332


In [ ]:
maelasoo = mae_metric(y_true=y_train, y_pred=l1_train_pred)
libaelasso = mae_metric(y_true=y_train, y_pred=sklearn_lasso_pred_train)
print(maelasoo)
print(libaelasso)

711.299151002065
711.3975281247201


In [ ]:
rmselasoo = rmse_metric(y_true=y_train, y_pred=l1_train_pred)
librmselasso = rmse_metric(y_true=y_train, y_pred=sklearn_lasso_pred_train)
print(rmselasoo)
print(librmselasso)

1035.364806845126
1035.548525823977


In [ ]:
r2lasso_test = r2_metric(y_true=y_test, y_pred=l1_test_pred)
libr2lasso_test = r2_metric(y_true=y_test, y_pred=sklearn_lasso_pred_test)
print(r2lasso_test)
print(libr2lasso_test)

0.5800231726410873
0.5798741149197332


In [ ]:
maelasso_test = mae_metric(y_true=y_test, y_pred=l1_test_pred)
librmae_test = mae_metric(y_true=y_test, y_pred=sklearn_lasso_pred_test)
print(maelasso_test)
print(librmae_test)

711.299151002065
711.3975281247201


In [ ]:
rmselasso_test = rmse_metric(y_true=y_test, y_pred=l1_test_pred)
librrmse_test = rmse_metric(y_true=y_test, y_pred=sklearn_lasso_pred_test)
print(rmselasso_test)
print(librrmse_test)
result_r2.loc[3] = ['Lasso', r2lasoo, r2_metric(y_test, l1_test_pred)]
result_mae.loc[3] = ['Lasso', maelasoo, mae_metric(y_test, l1_test_pred)]
result_rmse.loc[3] = ['Lasso', rmselasoo, rmse_metric(y_test, l1_test_pred)]
result_rmse

1035.364806845126
1035.548525823977


,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807


In [ ]:
reg_model_el = MyLinearRegressionRegulized(algorithm='elasticnet')
reg_model_el.fit(X_train, y_train)
el_train_pred = reg_model_el.predict(X_train)
el_test_pred = reg_model_el.predict(X_test)
from sklearn.linear_model import ElasticNet
sklearn_el = ElasticNet()
sklearn_el.fit(X_train, y_train)
sklearn_el_pred_train = sklearn_el.predict(X_train)
sklearn_el_pred_test = sklearn_el.predict(X_test)

In [ ]:
r2_el = r2_metric(y_true=y_train, y_pred=el_train_pred)
libr2el = r2_metric(y_true=y_train, y_pred=sklearn_el_pred_train)
print(r2_el)
print(libr2el)

0.5684111396679281
0.4452712967111435


In [ ]:
mae_el = mae_metric(y_true=y_train, y_pred=el_train_pred)
libmaeel = mae_metric(y_true=y_train, y_pred=sklearn_el_pred_train)
print(mae_el)
print(libmaeel)

716.2579130649135
807.0916753510556


In [ ]:
rmse_el = rmse_metric(y_true=y_train, y_pred=el_train_pred)
lib_rmse_el = rmse_metric(y_true=y_train, y_pred=sklearn_el_pred_train)
print(rmse_el)
print(lib_rmse_el)

1049.5807282931612
1189.9290128312373


In [ ]:
r2_el_test = r2_metric(y_true=y_test, y_pred=el_test_pred)
libr2el_test = r2_metric(y_true=y_test, y_pred=sklearn_el_pred_test)
print(r2_el_test)
print(libr2el_test)

0.5684111396679281
0.4452712967111435


In [ ]:
mae_el_test = mae_metric(y_true=y_test, y_pred=el_test_pred)
librmael_test = mae_metric(y_true=y_test, y_pred=sklearn_el_pred_test)
print(r2_el_test)
print(librmael_test)

0.5684111396679281
807.0916753510556


In [ ]:
rmse_el_test = rmse_metric(y_true=y_test, y_pred=el_test_pred)
librmsel_test = rmse_metric(y_true=y_test, y_pred=sklearn_el_pred_test)
print(rmse_el_test)
print(librmsel_test)

1049.5807282931612
1189.9290128312373


In [ ]:
result_r2.loc[4] = ['ElasticNet', r2_el, r2_el_test]
result_mae.loc[4] = ['ElasticNet', mae_el, mae_el_test]
result_rmse.loc[4] = ['ElasticNet', rmse_el, rmse_el_test]

In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728


## Нормализация признаков

Нормализация признаков - это преобразование данных так, чтобы все признаки были в одном масштабе, например, от -1 до 1.

Нормализация нужна, где признаки имеют разные масштабы, т.е. в градиентных методах(Линейная, логистическая регрессия, нейронные сети), если ее не применить, то gd будет шагать неравномерно, из-за этого модель будет дольше и нестабильно обучаться. Деревьям и ансамблям она не нужна, т.к.  там данные делятся по порогам

### Метод №1 MinMaxScaler

$$
x_{\text{scaled}} = \frac{x-x_{\min}}{x_{\max} - x_{\min}}
$$

In [ ]:
class MyMinMaxScaler():
    def fit(self, X):
        self.min_ = X.min(axis=0)
        self.max_ = X.max(axis=0)
        return self
    
    def transform(self, X):
        return (X - self.min_) / (self.max_ - self.min_)
    
    def fit_transform(self, X):
        self.fit(X)
        return np.array(self.transform(X))

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

my_scaler = MyMinMaxScaler()
X_scaled_my = my_scaler.fit_transform(X_train)
X_scaled_my_test = my_scaler.transform(X_test)

y_scaled_my = y_train.values
y_scaled_my_test = y_test.values

sk_scaler = MinMaxScaler()
X_scaled_sk = sk_scaler.fit_transform(X_train)

pd.DataFrame(y_scaled_my)

,0
0,2400
1,3800
2,3495
3,3000
4,2795
...,...
48374,2800
48375,2395
48376,1850
48377,4195


In [ ]:
pd.DataFrame(X_scaled_sk)

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,0.10,0.125,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.10,0.250,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.10,0.250,1.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.15,0.375,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.10,0.000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48374,0.10,0.375,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
48375,0.10,0.250,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
48376,0.10,0.125,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
48377,0.10,0.250,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Метод №2 StandartScaler

$$
x_{\text{scaled}} = \frac{x-\mu}{\sigma}
$$

где $\mu$-среднее значение признака, $\sigma$-стандартное отклонение

In [ ]:
class MyStandartScaler():
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_ = X.std(axis=0)
        self.std_[self.std_ == 0] = 1
        return self
    
    def transform(self, X):
        return (X - self.mean_) / self.std_
    
    def fit_transform(self, X):
        self.fit(X)
        return self.transform(X)

In [ ]:
my_standart_scaler = MyStandartScaler()
X_scaled_st_my = my_standart_scaler.fit_transform(X_train)
X_scaled_st_my_test = my_standart_scaler.transform(X_test)

y_scaled_st_my = y_train.values
y_scaled_st_my_test = y_test.values

sk_standart_scaler = StandardScaler()
X_scaled_st_sk = sk_standart_scaler.fit_transform(X_train)

pd.DataFrame(X_scaled_st_my)


,bathrooms,bedrooms,Elevator,HardwoodFloors,CatsAllowed,DogsAllowed,Doorman,Dishwasher,NoFee,LaundryinBuilding,...,LaundryinUnit,RoofDeck,OutdoorSpace,DiningRoom,HighSpeedInternet,Balcony,SwimmingPool,LaundryInBuilding,NewConstruction,Terrace
0,-0.427602,-0.485378,-1.051262,1.043538,1.044013,1.110555,-0.857391,1.186375,-0.763150,1.416394,...,-0.459957,-0.391262,-0.344665,2.978434,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
1,-0.427602,0.422494,0.951218,1.043538,-0.957822,-0.900432,1.166305,1.186375,1.310332,1.416394,...,-0.459957,-0.391262,-0.344665,-0.335740,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
2,-0.427602,0.422494,0.951218,1.043538,-0.957822,-0.900432,1.166305,1.186375,-0.763150,1.416394,...,2.174071,-0.391262,-0.344665,-0.335740,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
3,0.667699,1.330365,-1.051262,-0.958259,-0.957822,-0.900432,-0.857391,-0.842887,-0.763150,-0.706004,...,-0.459957,-0.391262,-0.344665,-0.335740,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
4,-0.427602,-1.393250,0.951218,-0.958259,-0.957822,-0.900432,1.166305,-0.842887,-0.763150,1.416394,...,-0.459957,-0.391262,-0.344665,-0.335740,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48374,-0.427602,1.330365,0.951218,1.043538,-0.957822,-0.900432,-0.857391,1.186375,-0.763150,-0.706004,...,-0.459957,-0.391262,-0.344665,-0.335740,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
48375,-0.427602,0.422494,0.951218,-0.958259,1.044013,1.110555,1.166305,-0.842887,1.310332,-0.706004,...,-0.459957,-0.391262,-0.344665,-0.335740,-0.309331,-0.252423,-0.24063,4.226209,-0.233775,-0.217172
48376,-0.427602,-0.485378,0.951218,1.043538,1.044013,1.110555,-0.857391,1.186375,1.310332,1.416394,...,2.174071,-0.391262,-0.344665,2.978434,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172
48377,-0.427602,0.422494,-1.051262,-0.958259,-0.957822,-0.900432,-0.857391,1.186375,1.310332,-0.706004,...,2.174071,-0.391262,2.901307,-0.335740,-0.309331,-0.252423,-0.24063,-0.236614,-0.233775,-0.217172


In [ ]:
pd.DataFrame(X_scaled_st_sk)

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,-0.427606,-0.485383,-1.051272,1.043549,1.044024,1.110567,-0.857399,1.186387,-0.763157,1.416409,...,-0.459962,-0.391266,-0.344669,2.978464,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
1,-0.427606,0.422498,0.951228,1.043549,-0.957832,-0.900441,1.166318,1.186387,1.310346,1.416409,...,-0.459962,-0.391266,-0.344669,-0.335743,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
2,-0.427606,0.422498,0.951228,1.043549,-0.957832,-0.900441,1.166318,1.186387,-0.763157,1.416409,...,2.174093,-0.391266,-0.344669,-0.335743,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
3,0.667706,1.330379,-1.051272,-0.958269,-0.957832,-0.900441,-0.857399,-0.842895,-0.763157,-0.706011,...,-0.459962,-0.391266,-0.344669,-0.335743,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
4,-0.427606,-1.393264,0.951228,-0.958269,-0.957832,-0.900441,1.166318,-0.842895,-0.763157,1.416409,...,-0.459962,-0.391266,-0.344669,-0.335743,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48374,-0.427606,1.330379,0.951228,1.043549,-0.957832,-0.900441,-0.857399,1.186387,-0.763157,-0.706011,...,-0.459962,-0.391266,-0.344669,-0.335743,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
48375,-0.427606,0.422498,0.951228,-0.958269,1.044024,1.110567,1.166318,-0.842895,1.310346,-0.706011,...,-0.459962,-0.391266,-0.344669,-0.335743,-0.309334,-0.252426,-0.240632,4.226252,-0.233778,-0.217174
48376,-0.427606,-0.485383,0.951228,1.043549,1.044024,1.110567,-0.857399,1.186387,1.310346,1.416409,...,2.174093,-0.391266,-0.344669,2.978464,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174
48377,-0.427606,0.422498,-1.051272,-0.958269,-0.957832,-0.900441,-0.857399,1.186387,1.310346,-0.706011,...,2.174093,-0.391266,2.901337,-0.335743,-0.309334,-0.252426,-0.240632,-0.236616,-0.233778,-0.217174


In [ ]:
lin_reg = MyLinearRegression()
lin_reg.fit(X_scaled_my, y_scaled_my)
train_pred = lin_reg.predict(X_scaled_my)
test_pred = lin_reg.predict(X_scaled_my_test)
result_r2.loc[5] = ['Linear MinMaxScaler', r2_metric(y_scaled_my, train_pred), r2_metric(y_scaled_my_test, test_pred)]
result_mae.loc[5] = ['Linear MinMaxScaler', mae_metric(y_scaled_my, train_pred), mae_metric(y_scaled_my_test, test_pred)]
result_rmse.loc[5] = ['Linear MinMaxScaler', rmse_metric(y_scaled_my, train_pred), rmse_metric(y_scaled_my_test, test_pred)]

In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411
5,Linear MinMaxScaler,0.529158,0.529158


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913
5,Linear MinMaxScaler,753.382010,753.382010


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728
5,Linear MinMaxScaler,1096.272432,1096.272432


In [ ]:
ridge_linreg = MyLinearRegressionRegulized()
ridge_linreg.fit(X_scaled_my, y_scaled_my)
ridge_train_pred = ridge_linreg.predict(X_scaled_my)
ridge_test_pred = ridge_linreg.predict(X_scaled_my_test)
result_r2.loc[6] = ['Ridge MinMaxScaler', r2_metric(y_scaled_my, ridge_train_pred), r2_metric(y_scaled_my_test, ridge_test_pred)]
result_mae.loc[6] = ['Ridge MinMaxScaler', mae_metric(y_scaled_my, ridge_train_pred), mae_metric(y_scaled_my_test, ridge_test_pred)]
result_rmse.loc[6] = ['Ridge MinMaxScaler', rmse_metric(y_scaled_my, ridge_train_pred), rmse_metric(y_scaled_my_test, ridge_test_pred)]

In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411
5,Linear MinMaxScaler,0.529158,0.529158
6,Ridge MinMaxScaler,0.238727,0.238727


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913
5,Linear MinMaxScaler,753.382010,753.382010
6,Ridge MinMaxScaler,979.156744,979.156744


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728
5,Linear MinMaxScaler,1096.272432,1096.272432
6,Ridge MinMaxScaler,1393.962024,1393.962024


In [ ]:
lasso_linreg = MyLinearRegressionRegulized(algorithm='l1')
lasso_linreg.fit(X_scaled_my, y_scaled_my)
lasso_train_pred = lasso_linreg.predict(X_scaled_my)
lasso_test_pred = lasso_linreg.predict(X_scaled_my_test)
result_r2.loc[7] = ['Lasso MinMaxScaler', r2_metric(y_scaled_my, lasso_train_pred), r2_metric(y_scaled_my_test, lasso_test_pred)]
result_mae.loc[7] = ['Lasso MinMaxScaler', mae_metric(y_scaled_my, lasso_train_pred), mae_metric(y_scaled_my_test, lasso_test_pred)]
result_rmse.loc[7] = ['Lasso MinMaxScaler', rmse_metric(y_scaled_my, lasso_train_pred), rmse_metric(y_scaled_my_test, lasso_test_pred)]

In [ ]:
el_linreg = MyLinearRegressionRegulized(algorithm='elasticnet')
el_linreg.fit(X_scaled_my, y_scaled_my)
el_train_pred = el_linreg.predict(X_scaled_my)
el_test_pred = el_linreg.predict(X_scaled_my_test)
result_r2.loc[8] = ['ElasticNet MinMaxScaler', r2_metric(y_scaled_my, el_train_pred), r2_metric(y_scaled_my_test, el_test_pred)]
result_mae.loc[8] = ['ElasticNet MinMaxScaler', mae_metric(y_scaled_my, el_train_pred), mae_metric(y_scaled_my_test, el_test_pred)]
result_rmse.loc[8] = ['ElasticNet MinMaxScaler', rmse_metric(y_scaled_my, el_train_pred), rmse_metric(y_scaled_my_test, el_test_pred)]

In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411
5,Linear MinMaxScaler,0.529158,0.529158
6,Ridge MinMaxScaler,0.238727,0.238727
7,Lasso MinMaxScaler,0.529061,0.529061
8,ElasticNet MinMaxScaler,0.303481,0.303481


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913
5,Linear MinMaxScaler,753.382010,753.382010
6,Ridge MinMaxScaler,979.156744,979.156744
7,Lasso MinMaxScaler,753.390944,753.390944
8,ElasticNet MinMaxScaler,927.799247,927.799247


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728
5,Linear MinMaxScaler,1096.272432,1096.272432
6,Ridge MinMaxScaler,1393.962024,1393.962024
7,Lasso MinMaxScaler,1096.385275,1096.385275
8,ElasticNet MinMaxScaler,1333.359725,1333.359725


In [ ]:
lin_reg1 = MyLinearRegression()
lin_reg1.fit(X_scaled_st_my, y_scaled_st_my)
lr1_train_pred = lin_reg1.predict(X_scaled_st_my)
lr1_test_pred = lin_reg1.predict(X_scaled_st_my_test)
result_r2.loc[9] = ['Linear StandartScaler', r2_metric(y_scaled_st_my, lr1_train_pred), r2_metric(y_scaled_st_my_test, lr1_test_pred)]
result_mae.loc[9] = ['Linear StandartScaler', mae_metric(y_scaled_st_my, lr1_train_pred), mae_metric(y_scaled_st_my_test, lr1_test_pred)]
result_rmse.loc[9] = ['Linear StandartScaler', rmse_metric(y_scaled_st_my, lr1_train_pred), rmse_metric(y_scaled_st_my_test, lr1_test_pred)]

In [ ]:
ridge_linreg1 = MyLinearRegressionRegulized()
ridge_linreg1.fit(X_scaled_st_my, y_scaled_st_my)
ridge_linreg1_train_pred = ridge_linreg1.predict(X_scaled_st_my)
ridge1_test_pred = ridge_linreg1.predict(X_scaled_st_my_test)
result_r2.loc[10] = ['Ridge StandartScaler', r2_metric(y_scaled_st_my, ridge_linreg1_train_pred), r2_metric(y_scaled_st_my_test, ridge1_test_pred)]
result_mae.loc[10] = ['Ridge StandartScaler', mae_metric(y_scaled_st_my, ridge_linreg1_train_pred), mae_metric(y_scaled_st_my_test, ridge1_test_pred)]
result_rmse.loc[10] = ['Ridge StandartScaler', rmse_metric(y_scaled_st_my, ridge_linreg1_train_pred), rmse_metric(y_scaled_st_my_test, ridge1_test_pred)]

In [ ]:
laso_linreg1 = MyLinearRegressionRegulized(algorithm='l1')
laso_linreg1.fit(X_scaled_st_my, y_scaled_st_my)
lasso1_train_pred = laso_linreg1.predict(X_scaled_st_my)
lasso1_test_pred = laso_linreg1.predict(X_scaled_st_my_test)
result_r2.loc[11] = ['Lasso StandartScaler', r2_metric(y_scaled_st_my, lasso1_train_pred), r2_metric(y_scaled_st_my_test, lasso1_test_pred)]
result_mae.loc[11] = ['Lasso StandartScaler', mae_metric(y_scaled_st_my, lasso1_train_pred), mae_metric(y_scaled_st_my_test, lasso1_test_pred)]
result_rmse.loc[11] = ['Lasso StandartScaler', rmse_metric(y_scaled_st_my, lasso1_train_pred), rmse_metric(y_scaled_st_my_test, lasso1_test_pred)]

In [ ]:
el_lin_reg1 = MyLinearRegressionRegulized(algorithm='ElasticNet')
el_lin_reg1.fit(X_scaled_st_my, y_train)
el1_train_pred = el_lin_reg1.predict(X_scaled_st_my)
el1_test_pred = el_lin_reg1.predict(X_scaled_st_my_test)
result_r2.loc[12] = ['ElasticNet StandartScaler', r2_metric(y_scaled_st_my, el1_train_pred), r2_metric(y_scaled_st_my_test, el1_test_pred)]
result_mae.loc[12] = ['ElasticNet StandartScaler', mae_metric(y_scaled_st_my, el1_train_pred), mae_metric(y_scaled_st_my_test, el1_test_pred)]
result_rmse.loc[12] = ['ElasticNet StandartScaler', rmse_metric(y_scaled_st_my, el1_train_pred), rmse_metric(y_scaled_st_my_test, el1_test_pred)]

In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411
5,Linear MinMaxScaler,0.529158,0.529158
6,Ridge MinMaxScaler,0.238727,0.238727
7,Lasso MinMaxScaler,0.529061,0.529061
8,ElasticNet MinMaxScaler,0.303481,0.303481
9,Linear StandartScaler,0.579901,0.579901
10,Ridge StandartScaler,0.577875,0.577875


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913
5,Linear MinMaxScaler,753.382010,753.382010
6,Ridge MinMaxScaler,979.156744,979.156744
7,Lasso MinMaxScaler,753.390944,753.390944
8,ElasticNet MinMaxScaler,927.799247,927.799247
9,Linear StandartScaler,712.699045,712.699045
10,Ridge StandartScaler,712.717166,712.717166


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728
5,Linear MinMaxScaler,1096.272432,1096.272432
6,Ridge MinMaxScaler,1393.962024,1393.962024
7,Lasso MinMaxScaler,1096.385275,1096.385275
8,ElasticNet MinMaxScaler,1333.359725,1333.359725
9,Linear StandartScaler,1035.515081,1035.515081
10,Ridge StandartScaler,1038.009820,1038.009820


## Переобученные модели

In [ ]:
X_train = X_train[['bathrooms', 'bedrooms']]
X_test = X_test[['bathrooms', 'bedrooms']]
X_train.shape

(48379, 2)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(10)
poly.fit(X_train)
X_train = poly.transform(X_train)
X_test = poly.transform(X_test)
X_train.shape

(48379, 66)

In [ ]:
lin_reg = MyLinearRegression(learning_rate=0.000000000000000000000000000000000000000000000000000001)
lin_reg.fit(X_train, y_train)
train_pred = lin_reg.predict(X_train)
test_pred = lin_reg.predict(X_test)
result_r2.loc[13] = ['Linreg Polynomial', r2_metric(y_train, train_pred), r2_metric(y_test, test_pred)]
result_mae.loc[13] = ['Linreg Polynomial', mae_metric(y_train, train_pred), mae_metric(y_test, test_pred)]
result_rmse.loc[13] = ['Linreg Polynomial', rmse_metric(y_train, train_pred), rmse_metric(y_test, test_pred)]

In [ ]:
ridge_lin_reg = MyLinearRegressionRegulized(learning_rate=0.000000000000000000000000000000000000000000000000000001)
ridge_lin_reg.fit(X_train, y_train)
train_pred = ridge_lin_reg.predict(X_train)
test_pred = ridge_lin_reg.predict(X_test)
result_r2.loc[14] = ['Ridge Polynomial', r2_metric(y_train, train_pred), r2_metric(y_test, test_pred)]
result_mae.loc[14] = ['Ridge Polynomial', mae_metric(y_train, train_pred), mae_metric(y_test, test_pred)]
result_rmse.loc[14] = ['Ridge Polynomial', rmse_metric(y_train, train_pred), rmse_metric(y_test, test_pred)]

In [ ]:
lasso_lin_reg = MyLinearRegressionRegulized(learning_rate=0.000000000000000000000000000000000000000000000000000001,algorithm='lasso')
lasso_lin_reg.fit(X_train, y_train)
train_pred = lasso_lin_reg.predict(X_train)
test_pred = lasso_lin_reg.predict(X_test)
result_r2.loc[15] = ['Lasso Polynomial', r2_metric(y_train, train_pred), r2_metric(y_test, test_pred)]
result_mae.loc[15] = ['Lasso Polynomial', mae_metric(y_train, train_pred), mae_metric(y_test, test_pred)]
result_rmse.loc[15] = ['Lasso Polynomial', rmse_metric(y_train, train_pred), rmse_metric(y_test, test_pred)]

In [ ]:
el_lin_reg = MyLinearRegressionRegulized(learning_rate=0.000000000000000000000000000000000000000000000000000001, algorithm='ElasticNet')
el_lin_reg.fit(X_train, y_train)
train_pred = el_lin_reg.predict(X_train)
test_pred = el_lin_reg.predict(X_test)
result_r2.loc[16] = ['ElasticNet Polynomial', r2_metric(y_train, train_pred), r2_metric(y_test, test_pred)]
result_mae.loc[16] = ['ElasticNet Polynomial', mae_metric(y_train, train_pred), mae_metric(y_test, test_pred)]
result_rmse.loc[16] = ['ElasticNet Polynomial', rmse_metric(y_train, train_pred), rmse_metric(y_test, test_pred)]

In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411
5,Linear MinMaxScaler,0.529158,0.529158
6,Ridge MinMaxScaler,0.238727,0.238727
7,Lasso MinMaxScaler,0.529061,0.529061
8,ElasticNet MinMaxScaler,0.303481,0.303481
9,Linear StandartScaler,0.579901,0.579901
10,Ridge StandartScaler,0.577875,0.577875


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913
5,Linear MinMaxScaler,753.382010,753.382010
6,Ridge MinMaxScaler,979.156744,979.156744
7,Lasso MinMaxScaler,753.390944,753.390944
8,ElasticNet MinMaxScaler,927.799247,927.799247
9,Linear StandartScaler,712.699045,712.699045
10,Ridge StandartScaler,712.717166,712.717166


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728
5,Linear MinMaxScaler,1096.272432,1096.272432
6,Ridge MinMaxScaler,1393.962024,1393.962024
7,Lasso MinMaxScaler,1096.385275,1096.385275
8,ElasticNet MinMaxScaler,1333.359725,1333.359725
9,Linear StandartScaler,1035.515081,1035.515081
10,Ridge StandartScaler,1038.009820,1038.009820


In [ ]:
mean_target_train = np.full(y_train.shape, y_train.mean())
mean_target_test = np.full(y_test.shape, y_test.mean())
median_target_train = np.full(y_train.shape, y_train.median())
median_target_test = np.full(y_test.shape, y_test.median())
result_r2.loc[17] = ["Naive mean ", r2_metric(y_train, mean_target_train), r2_metric(y_test, mean_target_test)]
result_mae.loc[17] = ["Naive mean ", mae_metric(y_train, mean_target_train), mae_metric(y_test, mean_target_test)]
result_rmse.loc[17] = ["Naive mean ", rmse_metric(y_train, mean_target_train), rmse_metric(y_test, mean_target_test)]
result_r2.loc[18] = ["Naive median ", r2_metric(y_train, median_target_train), r2_metric(y_test, median_target_test)]
result_mae.loc[18] = ["Naive median ", mae_metric(y_train, median_target_train), mae_metric(y_test, median_target_test)]
result_rmse.loc[18] = ["Naive median ", rmse_metric(y_train, median_target_train), rmse_metric(y_test, median_target_test)]


In [ ]:
result_r2

,model,train,test
1,sgd,0.580024,0.580024
2,Ridge,0.550489,0.550489
3,Lasso,0.580023,0.580023
4,ElasticNet,0.568411,0.568411
5,Linear MinMaxScaler,0.529158,0.529158
6,Ridge MinMaxScaler,0.238727,0.238727
7,Lasso MinMaxScaler,0.529061,0.529061
8,ElasticNet MinMaxScaler,0.303481,0.303481
9,Linear StandartScaler,0.579901,0.579901
10,Ridge StandartScaler,0.577875,0.577875


In [ ]:
result_mae

,model,train,test
1,sgd,711.331442,711.331442
2,Ridge,728.216896,728.216896
3,Lasso,711.299151,711.299151
4,ElasticNet,716.257913,716.257913
5,Linear MinMaxScaler,753.382010,753.382010
6,Ridge MinMaxScaler,979.156744,979.156744
7,Lasso MinMaxScaler,753.390944,753.390944
8,ElasticNet MinMaxScaler,927.799247,927.799247
9,Linear StandartScaler,712.699045,712.699045
10,Ridge StandartScaler,712.717166,712.717166


In [ ]:
result_rmse

,model,train,test
1,sgd,1035.364272,1035.364272
2,Ridge,1071.150964,1071.150964
3,Lasso,1035.364807,1035.364807
4,ElasticNet,1049.580728,1049.580728
5,Linear MinMaxScaler,1096.272432,1096.272432
6,Ridge MinMaxScaler,1393.962024,1393.962024
7,Lasso MinMaxScaler,1096.385275,1096.385275
8,ElasticNet MinMaxScaler,1333.359725,1333.359725
9,Linear StandartScaler,1035.515081,1035.515081
10,Ridge StandartScaler,1038.009820,1038.009820


По метрикам лучше всего выгдядит модель Lasso, особенно при использовании MinMaxScaler